### 1. Title & Objective

# Sentiment Regime Analysis: Impact on Trader Behavior & Profitability

## Objective

Analyze how market sentiment regimes (Fear/Greed classification) influence:
- Trader profitability
- Trading intensity
- Position sizing
- Directional bias

The analysis is conducted at daily granularity after merging trade-level data with sentiment classifications.

# 2. Data Loading

In [49]:
import pandas as pd
import numpy as np

sentiment = pd.read_csv("fear_greed_index.csv")
trades = pd.read_csv("historical_data.csv")

In [50]:
trades.head()
sentiment.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


# 3. Initial Data Inspection

In [51]:
trades.info()
sentiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Account           211224 non-null  object 
 1   Coin              211224 non-null  object 
 2   Execution Price   211224 non-null  float64
 3   Size Tokens       211224 non-null  float64
 4   Size USD          211224 non-null  float64
 5   Side              211224 non-null  object 
 6   Timestamp IST     211224 non-null  object 
 7   Start Position    211224 non-null  float64
 8   Direction         211224 non-null  object 
 9   Closed PnL        211224 non-null  float64
 10  Transaction Hash  211224 non-null  object 
 11  Order ID          211224 non-null  int64  
 12  Crossed           211224 non-null  bool   
 13  Fee               211224 non-null  float64
 14  Trade ID          211224 non-null  float64
 15  Timestamp         211224 non-null  float64
dtypes: bool(1), float64(

# 4. Data Cleaning

In [4]:
sentiment.columns = sentiment.columns.str.lower().str.replace(" ", "_")
trades.columns = trades.columns.str.lower().str.replace(" ", "_")

In [6]:
sentiment['date'].head()

0    2018-02-01
1    2018-02-02
2    2018-02-03
3    2018-02-04
4    2018-02-05
Name: date, dtype: object

In [7]:
sentiment['date'] = pd.to_datetime(sentiment['date'])

In [8]:
sentiment['date'] = sentiment['date'].dt.normalize()

In [9]:
sentiment['date'].dtype

dtype('<M8[ns]')

In [10]:
trades['timestamp_ist'].head(10)

0    02-12-2024 22:50
1    02-12-2024 22:50
2    02-12-2024 22:50
3    02-12-2024 22:50
4    02-12-2024 22:50
5    02-12-2024 22:50
6    02-12-2024 22:50
7    02-12-2024 22:50
8    02-12-2024 22:50
9    02-12-2024 22:50
Name: timestamp_ist, dtype: object

In [11]:
trades['timestamp_ist'].dtype

dtype('O')

In [12]:
trades['datetime'] = pd.to_datetime(
    trades['timestamp_ist'],
    format='%d-%m-%Y %H:%M',
    errors='coerce'
)

In [13]:
trades['datetime'].isna().mean()

np.float64(0.0)

In [14]:
trades['date'] = trades['datetime'].dt.normalize()

In [15]:
trades[['datetime','date']].head()

,datetime,date
0,2024-12-02 22:50:00,2024-12-02
1,2024-12-02 22:50:00,2024-12-02
2,2024-12-02 22:50:00,2024-12-02
3,2024-12-02 22:50:00,2024-12-02
4,2024-12-02 22:50:00,2024-12-02


In [16]:
print("Sentiment range:", sentiment['date'].min(), "to", sentiment['date'].max())
print("Trades range:", trades['date'].min(), "to", trades['date'].max())

Sentiment range: 2018-02-01 00:00:00 to 2025-05-02 00:00:00
Trades range: 2023-05-01 00:00:00 to 2025-05-01 00:00:00


# 4.2 Merge Datasets

In [17]:
df = trades.merge(
    sentiment[['date', 'classification']],
    on='date',
    how='left'
)

In [20]:
df['classification'].isna().mean()

np.float64(2.840586297011703e-05)

In [21]:
df.columns

Index(['account', 'coin', 'execution_price', 'size_tokens', 'size_usd', 'side',
       'timestamp_ist', 'start_position', 'direction', 'closed_pnl',
       'transaction_hash', 'order_id', 'crossed', 'fee', 'trade_id',
       'timestamp', 'datetime', 'date', 'classification'],
      dtype='object')

In [22]:
df['side'].value_counts()

side
SELL    108528
BUY     102696
Name: count, dtype: int64

In [23]:
df['direction'].value_counts()

direction
Open Long                    49895
Close Long                   48678
Open Short                   39741
Close Short                  36013
Sell                         19902
Buy                          16716
Spot Dust Conversion           142
Short > Long                    70
Long > Short                    57
Auto-Deleveraging                8
Liquidated Isolated Short        1
Settlement                       1
Name: count, dtype: int64

In [24]:
df['is_long_open'] = (df['direction'] == 'Open Long').astype(int)
df['is_short_open'] = (df['direction'] == 'Open Short').astype(int)

In [25]:
daily = df.groupby(['account', 'date', 'classification']).agg(
    daily_pnl = ('closed_pnl', 'sum'),
    trades_count = ('closed_pnl', 'count'),
    avg_trade_size_usd = ('size_usd', 'mean'),
    long_opens = ('is_long_open', 'sum'),
    short_opens = ('is_short_open', 'sum')
).reset_index()

In [26]:
daily['long_ratio'] = daily['long_opens'] / (
    daily['long_opens'] + daily['short_opens']
)

In [27]:
daily['long_ratio'] = daily['long_ratio'].fillna(0.5)

In [28]:
daily['win_day'] = (daily['daily_pnl'] > 0).astype(int)

In [29]:
daily.head()
daily.describe()

,date,daily_pnl,trades_count,avg_trade_size_usd,long_opens,short_opens,long_ratio,win_day
count,2340,2340.000000,2340.000000,2340.000000,2340.000000,2340.000000,2340.000000,2340.000000
mean,2024-12-22 01:24:55.384615680,4382.259380,90.264103,6986.186847,21.322650,16.983333,0.508087,0.626496
min,2023-05-01 00:00:00,-358963.139984,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2024-11-28 00:00:00,0.000000,9.000000,692.696303,0.000000,0.000000,0.000000,0.000000
50%,2025-01-28 00:00:00,206.352695,29.000000,1913.761949,0.000000,0.000000,0.500000,1.000000
75%,2025-03-18 00:00:00,1842.763729,80.000000,7033.497292,10.000000,12.000000,1.000000,1.000000
max,2025-05-01 00:00:00,533974.662903,4083.000000,844654.190000,2675.000000,676.000000,1.000000,1.000000
std,NaN,28411.103383,214.650554,21542.693224,100.650538,49.646593,0.397510,0.483838


In [30]:
daily['total_opens'] = daily['long_opens'] + daily['short_opens']
daily['total_opens'].describe()

count    2340.000000
mean       38.305983
std       113.682487
min         0.000000
25%         1.000000
50%         9.000000
75%        32.000000
max      2675.000000
Name: total_opens, dtype: float64

In [31]:
daily['activity_intensity'] = daily['total_opens']

# Question 1: Does Sentiment Affect Profitability?

### 1. Performance by Sentiment Regime

In [39]:
performance_table = daily.groupby('classification')['daily_pnl'].agg(
    ['mean','median','std','count']
)

performance_table

,mean,median,std,count
classification,,,,
Extreme Fear,4619.439053,218.377399,29534.839183,160
Extreme Greed,5161.922644,418.319862,27496.863832,526
Fear,5328.818161,107.892532,31659.771538,630
Greed,3318.100730,158.214922,30599.040173,648
Neutral,3438.618818,167.551743,17447.863645,376


### 2. Win Rate by Sentiment

In [38]:
winrate_table = daily.groupby('classification')['win_day'].mean()

winrate_table

classification
Extreme Fear     0.600000
Extreme Greed    0.682510
Fear             0.604762
Greed            0.611111
Neutral          0.622340
Name: win_day, dtype: float64

### 3. Risk-Adjusted Performance (Mean / Std)

In [40]:
risk_table = daily.groupby('classification')['daily_pnl'].agg(['mean','std'])
risk_table['mean_std_ratio'] = risk_table['mean'] / risk_table['std']

risk_table

,mean,std,mean_std_ratio
classification,,,
Extreme Fear,4619.439053,29534.839183,0.156406
Extreme Greed,5161.922644,27496.863832,0.187728
Fear,5328.818161,31659.771538,0.168315
Greed,3318.100730,30599.040173,0.108438
Neutral,3438.618818,17447.863645,0.197080



### Key Insights
1. Return vs Consistency Tradeoff

Although average daily PnL is highest during Fear regimes (~5.3k), win rates peak during Extreme Greed (~68%). This suggests that Fear periods offer larger payoff magnitude driven by volatility, while Extreme Greed provides more consistent but less dispersed returns. There is a clear tradeoff between payoff size and outcome consistency across sentiment regimes.

2. Risk Amplification in Fear

Fear regimes exhibit the highest PnL volatility (std ~31.7k), indicating amplified risk exposure. Elevated mean returns appear to be driven by a smaller number of large gains rather than uniformly improved daily performance.

3. Greed Underperformance

Contrary to intuitive expectations, regular Greed periods do not maximize trader profitability. Mean PnL during Greed (~3.3k) is materially lower than during Fear, suggesting that optimism-driven environments may compress volatility and reduce exploitable price dislocations.

**Trader profitability appears higher during Fear regimes compared to Greed. This suggests that heightened volatility during fear-driven markets creates larger opportunity sets. However, standard deviation of PnL is also elevated, indicating increased risk exposure.**

# Question 2: How Does Sentiment Affect Trader Behavior and Profitability?

# 2.1 Data Aggregation (Daily Level Construction)

In [53]:
daily = df.groupby(['date','classification']).agg(
    daily_pnl=('closed_pnl','sum'),
    trades_count=('trade_id','count'),
    avg_trade_size_usd=('size_usd','mean'),
    long_opens=('direction', lambda x: (x=='Open Long').sum()),
    short_opens=('direction', lambda x: (x=='Open Short').sum())
).reset_index()

daily['total_opens'] = daily['long_opens'] + daily['short_opens']
daily['long_ratio'] = daily['long_opens'] / daily['total_opens']
daily['win_day'] = (daily['daily_pnl'] > 0).astype(int)

### 2.2 Activity Intensity by Sentiment

In [41]:
activity_table = daily.groupby('classification')['total_opens'].agg(
    ['mean','median','std','count']
)

activity_table

,mean,median,std,count
classification,,,,
Extreme Fear,63.618750,17.0,158.244134,160
Extreme Greed,26.545627,8.0,53.660811,526
Fear,45.573016,8.0,141.383535,630
Greed,31.185185,8.0,72.340768,648
Neutral,44.082447,8.0,150.750907,376


### 2.3 Average Trade Size by Sentiment

In [42]:
size_table = daily.groupby('classification')['avg_trade_size_usd'].agg(
    ['mean','median','std']
)

size_table

,mean,median,std
classification,,,
Extreme Fear,6773.464125,2315.629870,11324.056422
Extreme Greed,5371.637182,2003.480176,8185.832780
Fear,8975.928546,1752.677497,36763.961226
Greed,6427.866594,2052.534828,12093.029200
Neutral,6963.694861,1704.405417,14705.124259


### 2.4 Directional Bias (Long Ratio)

In [43]:
direction_table = daily.groupby('classification')['long_ratio'].agg(
    ['mean','median','std']
)

direction_table

,mean,median,std
classification,,,
Extreme Fear,0.624988,0.68246,0.400349
Extreme Greed,0.486918,0.50000,0.377531
Fear,0.542244,0.50000,0.407689
Greed,0.473962,0.50000,0.394741
Neutral,0.489534,0.50000,0.399594


# 2.5 Profitability by Sentiment

In [54]:
pnl_table = daily.groupby('classification')['daily_pnl'].agg(
    ['mean','median','std','count']
)
pnl_table

,mean,median,std,count
classification,,,,
Extreme Fear,52793.589178,22561.739636,101262.394065,14
Extreme Greed,23817.292199,3127.536297,72827.301581,114
Fear,36891.818040,1412.314654,96611.848503,91
Greed,11140.566181,678.475928,62427.957949,193
Neutral,19297.323516,1818.573295,37995.209071,67


# 2.6 Consistency Check (Win Rate)

In [55]:
winrate_table = daily.groupby('classification')['win_day'].mean()
winrate_table

classification
Extreme Fear     0.642857
Extreme Greed    0.877193
Fear             0.736264
Greed            0.725389
Neutral          0.671642
Name: win_day, dtype: float64

# 2.7 Risk-Adjusted Performance Check (IMPORTANT)

In [56]:
risk_adjusted = pnl_table['mean'] / pnl_table['std']
risk_adjusted

classification
Extreme Fear     0.521354
Extreme Greed    0.327038
Fear             0.381856
Greed            0.178455
Neutral          0.507888
dtype: float64

*The absence of clear regime dominance on a risk-adjusted basis suggests that sentiment primarily influences variance and trader behavior rather than pure alpha generation.*

## Behavioral Insights
# 1. Fear Regimes Trigger Aggressive Trading Activity
# Evidence
* Mean daily opens:
    - Extreme Fear: 63.6
    - Fear: 45.6
    - Greed: 31.2
    - Extreme Greed: 26.5
* Median opens remain ~8 across regimes (high skew)

# Interpretation
Trading activity increases sharply during Fear regimes, particularly Extreme Fear, where average opens are more than 2x those observed during Extreme Greed. However, the flat median suggests this increase is driven by a subset of highly active traders rather than uniform behavioral change across all participants.

# Implication
Fear-induced volatility appears to activate high-frequency or opportunistic traders, leading to concentration of activity rather than broad-based participation.

# 2. Position Sizing Expands During Fear Regimes

# Evidence
* Mean trade size (USD):
    - Fear: 8,976
    - Extreme Fear: 6,773
    - Greed: 6,428
    - Extreme Greed: 5,372

* Standard deviation of trade size is highest during Fear (high dispersion)

# Interpretation
Average trade size peaks during Fear regimes, indicating that traders increase capital deployment per trade during negative sentiment periods. The elevated dispersion suggests that some traders scale aggressively, amplifying exposure in volatile environments.

# Implication
Fear does not lead to de-risking behavior. Instead, traders increase both activity and position size, reflecting heightened risk appetite and volatility-driven opportunity seeking.

# 3. Stronger Long Bias During Fear (Dip-Buying Behavior)

# Evidence
* Mean long ratio:
    - Extreme Fear: 0.625
    - Fear: 0.542
    - Greed: 0.474
    - Extreme Greed: 0.487
    - Neutral: 0.490

* Median long ratio ≈ 0.50 across regimes

# Interpretation
Traders exhibit a stronger long bias during Fear regimes, particularly Extreme Fear. Contrary to expectations of increased shorting during market stress, positioning tilts toward long exposure, suggesting dip-buying behavior rather than panic-driven bearish positioning.

# Implication
Negative sentiment periods are treated as mean-reversion opportunities. Traders appear to anticipate rebounds during Fear regimes, increasing long exposure instead of aligning with downward momentum.

# Strategy Recommendations
# 1. Volatility-Responsive Risk Scaling During Fear

# Rationale
Fear regimes are associated with:
- Higher trading intensity
- Larger position sizes
- Higher PnL volatility
- Stronger long bias

While average returns are elevated, risk dispersion is significantly higher.

# Strategy Rule
During Fear and Extreme Fear regimes:
    - Cap position size at 75% of trader’s rolling 30-day average size
    - Limit total daily opens to a predefined threshold (e.g., 1.5x personal median)
    - Apply tighter stop-loss levels relative to recent volatility

# Expected Benefit
This controls volatility amplification while preserving participation in high-opportunity regimes. It reduces tail-risk exposure driven by aggressive scaling during sentiment shocks.

# 2. Consistency-Focused Participation During Extreme Greed

# Rationale
Extreme Greed exhibits:
- Highest win rate (~68%)
- Lower volatility dispersion than Fear
- Moderate activity levels

Returns are more consistent but less explosive.

# Strategy Rule
During Extreme Greed regimes:
    - Increase participation frequency selectively for traders with historical win rate > 60%
    - Maintain standard position sizing (avoid leverage amplification)
    - Favor systematic, rule-based entries over discretionary high-risk trades

# Expected Benefit
This leverages higher consistency environments while avoiding overconfidence-driven overtrading. The focus shifts from volatility capture to steady accumulation.

In [58]:
# Create smaller samples

trades_sample = trades.sample(n=5000, random_state=42)
sentiment_sample = sentiment.copy()  # sentiment usually small already

# Save to CSV inside data folder
trades_sample.to_csv("trades_sample.csv", index=False)
sentiment_sample.to_csv("sentiment_sample.csv", index=False)